# bonus term — E[bonus] as a contemporaneous scoring-map

**Render-not-decide.** A re-runnable view of the `bonus` term, **not** the record — the frozen numbers live in [predictive-phase3-points-model.md](../../../docs/studies/results/predictive-phase3-points-model.md). Logic is in `model/terms/bonus/`.

The **only non-forecast** term: bonus is caused by the same-match performance, so it is a per-position **OLS calibration** of realized bonus on `returns_pts` (the FPL value of the modelled returns — D-B's strong BPS proxy). It exposes intercept/slope so the simulator applies bonus per draw. See `ASSUMPTIONS.md`.

## Setup

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart
from model.terms.bonus import BonusModel, BonusTerm

mart = load_mart()
term = BonusTerm(BonusModel())

## Pre-fit assumptions
> OLS calibration: report the proxy strength (returns_pts vs realized bonus). See `ASSUMPTIONS.md` §3.

In [ ]:
report = term.model.check_assumptions(BonusModel.population(mart))
print(f"detectable={report.detectable}  n_train={report.n_train}")
report.dispersion  # family=gaussian_ols / proxy_spearman

## Gate — E[bonus] vs the returns_pts signal (calibration, not a ranking win)
> The proxy preserves the returns_pts ranking by construction, so proxy rho == signal rho. The fit sets magnitude; the gate confirms nothing regressed.

In [ ]:
gate = term.validate(mart)
display(gate.table)

## Calibration coefficients (for the simulator) + magnitude check
> The per-(position, gw) slope/intercept the simulator uses to co-move bonus with sampled returns; plus E[bonus] vs realized by returns_pts level.

In [ ]:
diag = term.diagnose(mart)
display(diag.ablation.groupby('position')[['intercept', 'slope']].mean())  # avg calibration per position

fitted = term.model.fit(mart)
pop = BonusModel.population(mart).assign(e_bonus=fitted.predictions)
ev = pop[pop['e_bonus'].notna()].copy()
ev['rp_bucket'] = ev['returns_pts'].round()
mag = ev.groupby('rp_bucket').agg(expected=('e_bonus', 'mean'), realized=('bonus_actual', 'mean'), n=('bonus_actual', 'size'))
ax = mag.plot(y=['expected', 'realized'], marker='o', figsize=(6, 4), title='bonus magnitude vs returns_pts')
ax.set_xlabel('returns_pts'); ax.set_ylabel('bonus'); plt.tight_layout()